In [1]:
import pandas as pd

df = pd.read_csv("../data/interim/packets_clean.csv")

df.head()

,timestamp_raw,timestamp_display,ip_version,src_ip,dst_ip,src_port,dst_port,protocol,packet_length,tcp_flags,is_broadcast,is_multicast,highest_layer
0,1.741991e+09,58.8475,4,192.168.0.22,192.168.0.255,47424,32761,UDP,116,NONE,False,False,Raw
1,1.741991e+09,62.8455,4,192.168.0.22,192.168.0.255,47424,32761,UDP,116,NONE,False,False,Raw
2,1.741991e+09,62.8535,4,192.168.0.22,192.168.0.255,47424,32761,UDP,116,NONE,False,False,Raw
3,1.741991e+09,89.5908,6,fe80::10fb:90d2:20ae:b5cd,ff02::1:2,546,547,UDP,126,NONE,False,True,DHCP6 Identity Association for Non-temporary A...
4,1.741991e+09,21.0305,4,192.168.0.38,192.168.0.255,57621,57621,UDP,72,NONE,False,False,Raw


In [2]:
df["timestamp_raw"] = pd.to_numeric(df["timestamp_raw"], errors="coerce")
df = df.sort_values("timestamp_raw")

In [3]:
df["packet_length"].describe()

count    7647.000000
mean      205.487904
std       295.562717
min        32.000000
25%        52.000000
50%        80.000000
75%       153.000000
max      1500.000000
Name: packet_length, dtype: float64

In [4]:
def is_local(ip):
    return ip.startswith("192.168")

df["src_local"] = df["src_ip"].apply(is_local)
df["dst_local"] = df["dst_ip"].apply(is_local)

df["direction"] = df.apply(
    lambda row: "outbound" if row["src_local"] else "inbound",
    axis=1
)

In [5]:
df["flow"] = (
    df["src_ip"] + "_" +
    df["dst_ip"] + "_" +
    df["protocol"] + "_" +
    df["dst_port"].astype(str)
)

In [7]:
flow_counts = df["flow"].value_counts()

flow_counts.head(10)

flow
192.168.0.22_192.168.0.53_UDP_58065      853
192.168.0.53_192.168.0.22_UDP_37152      746
192.168.0.53_192.168.0.22_UDP_43117      656
192.168.0.53_52.12.120.254_TCP_443       580
52.12.120.254_192.168.0.53_TCP_46692     403
192.168.0.53_216.105.169.18_UDP_10001    239
192.168.0.53_108.181.62.233_UDP_10001    238
192.168.0.53_108.181.23.175_UDP_10001    236
192.168.0.53_108.181.55.129_UDP_10001    234
54.148.40.77_192.168.0.203_TCP_60524     227
Name: count, dtype: int64

In [8]:
flow_stats = df.groupby("flow")["packet_length"].agg([
    "count", "mean", "std"
])

flow_stats.head()

,count,mean,std
flow,,,
100.21.159.71_192.168.0.53_TCP_50402,1,52.000000,NaN
100.21.159.71_192.168.0.53_TCP_50407,2,56.000000,5.656854
100.21.159.71_192.168.0.53_TCP_50419,1,60.000000,NaN
100.21.159.71_192.168.0.53_TCP_50469,2,56.000000,5.656854
108.181.23.175_192.168.0.53_UDP_58065,59,92.152542,44.472314


In [9]:
df["time_diff"] = df.groupby("flow")["timestamp_raw"].diff()

df["time_diff"].describe()

count    7339.000000
mean       14.282460
std        85.631559
min         0.000000
25%         0.004002
50%         0.096049
75%         3.061530
max      2248.140819
Name: time_diff, dtype: float64

In [14]:
flow_counts.head(10)

flow
192.168.0.22_192.168.0.53_UDP_58065      853
192.168.0.53_192.168.0.22_UDP_37152      746
192.168.0.53_192.168.0.22_UDP_43117      656
192.168.0.53_52.12.120.254_TCP_443       580
52.12.120.254_192.168.0.53_TCP_46692     403
192.168.0.53_216.105.169.18_UDP_10001    239
192.168.0.53_108.181.62.233_UDP_10001    238
192.168.0.53_108.181.23.175_UDP_10001    236
192.168.0.53_108.181.55.129_UDP_10001    234
54.148.40.77_192.168.0.203_TCP_60524     227
Name: count, dtype: int64

In [13]:
flow_stats.head()

,count,mean,std
flow,,,
100.21.159.71_192.168.0.53_TCP_50402,1,52.000000,NaN
100.21.159.71_192.168.0.53_TCP_50407,2,56.000000,5.656854
100.21.159.71_192.168.0.53_TCP_50419,1,60.000000,NaN
100.21.159.71_192.168.0.53_TCP_50469,2,56.000000,5.656854
108.181.23.175_192.168.0.53_UDP_58065,59,92.152542,44.472314


In [12]:
df["time_diff"].describe()

count    7339.000000
mean       14.282460
std        85.631559
min         0.000000
25%         0.004002
50%         0.096049
75%         3.061530
max      2248.140819
Name: time_diff, dtype: float64

In [15]:
flow_features = df.groupby("flow").agg({
    "packet_length": ["count", "mean", "std"],
    "time_diff": ["mean", "std"],
    "protocol": "first",
    "dst_port": "first"
})

flow_features.columns = [
    "packet_count",
    "packet_size_mean",
    "packet_size_std",
    "time_mean",
    "time_std",
    "protocol",
    "dst_port"
]

flow_features = flow_features.reset_index()

flow_features.head()

,flow,packet_count,packet_size_mean,packet_size_std,time_mean,time_std,protocol,dst_port
0,100.21.159.71_192.168.0.53_TCP_50402,1,52.000000,NaN,NaN,NaN,TCP,50402
1,100.21.159.71_192.168.0.53_TCP_50407,2,56.000000,5.656854,0.072036,NaN,TCP,50407
2,100.21.159.71_192.168.0.53_TCP_50419,1,60.000000,NaN,NaN,NaN,TCP,50419
3,100.21.159.71_192.168.0.53_TCP_50469,2,56.000000,5.656854,0.008000,NaN,TCP,50469
4,108.181.23.175_192.168.0.53_UDP_58065,59,92.152542,44.472314,45.796654,68.77181,UDP,58065


In [ ]:
# Anomaly Detection
features = flow_features[[
    "packet_count",
    "packet_size_mean",
    "time_mean"
]].fillna(0)

In [19]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(random_state=42)
flow_features["anomaly"] = model.fit_predict(features)

In [20]:
flow_features["anomaly"].value_counts()

anomaly
 1    55
-1    13
Name: count, dtype: int64

In [21]:
anomalies = flow_features[flow_features["anomaly"] == -1]

anomalies.head(10)

,flow,packet_count,packet_size_mean,packet_size_std,time_mean,time_std,protocol,dst_port,anomaly
11,18.214.19.97_192.168.0.199_TCP_52540,17,645.764706,633.050011,0.280000,0.856072,TCP,52540,-1
85,192.168.0.1_239.255.255.250_UDP_1900,143,520.699301,29.896072,12.256653,102.865622,UDP,1900,-1
95,192.168.0.22_192.168.0.53_UDP_58065,853,155.008206,171.152552,0.250731,4.796764,UDP,58065,-1
97,192.168.0.232_192.168.0.255_UDP_57621,25,72.000000,0.000000,114.321309,272.276409,UDP,57621,-1
105,192.168.0.53_100.21.159.71_TCP_443,11,54.181818,3.736795,230.048460,515.909669,TCP,443,-1
110,192.168.0.53_192.168.0.22_UDP_37152,746,473.087131,313.701648,0.021487,0.036639,UDP,37152,-1
111,192.168.0.53_192.168.0.22_UDP_43117,656,426.439024,314.841580,0.018599,0.033670,UDP,43117,-1
118,192.168.0.53_34.218.162.3_TCP_443,11,54.909091,4.036200,147.893533,361.295189,TCP,443,-1
121,192.168.0.53_35.80.233.132_TCP_443,98,608.010204,625.850423,26.603932,121.264240,TCP,443,-1
130,192.168.0.53_44.235.109.71_TCP_8883,15,66.466667,16.008331,192.984128,253.597473,TCP,8883,-1


In [23]:
#Finger Printing
device_features = df.groupby("src_ip").agg({
    "packet_length": ["mean", "std"],
    "time_diff": ["mean", "std"],
    "dst_port": "nunique"
})

device_features.columns = [
    "pkt_size_mean",
    "pkt_size_std",
    "time_mean",
    "time_std",
    "unique_ports"
]

device_features = device_features.reset_index()

In [ ]:
device_features.sort_values("unique_ports", ascending=False).head()

,src_ip,pkt_size_mean,pkt_size_std,time_mean,time_std,unique_ports
66,68.105.28.11,224.216216,57.630678,0.072037,0.096215,72
11,192.168.0.116,244.783550,344.957724,22.967053,95.994074,29
14,192.168.0.199,241.927273,327.308156,33.691837,125.752789,26
21,192.168.0.53,226.034036,306.844138,10.975600,82.221784,8
31,35.80.233.132,340.328571,514.862321,0.058615,0.095674,7


In [ ]:
#Visualization 
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(
    flow_features["packet_size_mean"],
    flow_features["time_mean"]
)
plt.xlabel("Packet Size Mean")
plt.ylabel("Time Mean")
plt.title("Flow Behavior")
plt.show()